# GenAI Client Practice Notebook

Use this notebook as a fast, beginner-friendly walkthrough of the `openai_sdk/genai_client` examples.

## Features covered
- Basic Responses API call
- Streaming text deltas
- Structured output parsing
- Function calling + API state continuation
- Built-in web search tool
- Reasoning settings inspection
- Optional multimodal and advanced tools


## 1. Setup

Before running cells:
1. Ensure `sandbox.yaml` is correctly configured (OCI profile, project, compartment).
2. Install dependencies with `uv sync` (or your project standard install step).
3. `.env` is optional for this notebook.

Use script pattern from project root when needed:
- `uv run python -m openai_sdk.genai_client.<script_name_without_py>`


In [ ]:
import json
from pathlib import Path

from openai import OpenAI
from pydantic import BaseModel

from openai_sdk.openai_client_provider import OpenAIClientProvider

MODEL_ID = "openai.gpt-5.2"
print("Step 1/1: Initializing OCI-backed OpenAI client...")
client: OpenAI = OpenAIClientProvider().oci_openai_client
print("Client ready.")


## 2. Basic Response (`base_client.py`)

Start with a simple single-call prompt.


In [ ]:
print("Running basic Responses API call...")
basic_response = client.responses.create(
    model=MODEL_ID,
    input="When did the Roman Empire fall?",
)
print("Result:")
basic_response.output_text


## 3. Streaming Output (`streaming.py`)

Print deltas as tokens arrive.


In [ ]:
def stream_text(prompt: str, model: str = MODEL_ID) -> str:
    print("Starting streaming request...")
    stream = client.responses.create(model=model, input=prompt, stream=True)
    print("Streaming output:")
    final_parts: list[str] = []
    for chunk in stream:
        if chunk.type == "response.output_text.delta":
            print(chunk.delta, end="", flush=True)
            final_parts.append(chunk.delta)
    print("\nStreaming complete.")
    return "".join(final_parts)


In [ ]:
stream_text("Why is the sky blue?")


## 4. Structured Output (`structured_response.py`)

Parse model output directly into a typed schema.


In [ ]:
class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

print("Requesting structured output parse...")
structured = client.responses.parse(
    model="openai.gpt-4.1",
    input=[
        {"role": "system", "content": "Extract event information."},
        {"role": "user", "content": "Team sync with Ana and Luis next Tuesday."},
    ],
    text_format=CalendarEvent,
    store=False,
)

print("Parsed object:")
structured.output_parsed


## 5. Function Calling + State (`api_state.py`)

Run function-calling loop manually and continue using `previous_response_id`.


In [ ]:
FUNCTION_TOOLS = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "Get current weather for a city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name"},
            },
            "required": ["city"],
        },
    }
]

def get_weather(city: str) -> str:
    return f"The weather in {city} is sunny with 24C."

print("Step 1: Let model request tool calls...")
first = client.responses.create(
    model="openai.gpt-5.4",
    tools=FUNCTION_TOOLS,
    input="What's the weather in Paris today?",
)

function_outputs = []
for item in first.output:
    if item.type == "function_call" and item.name == "get_weather":
        args = json.loads(item.arguments)
        weather = get_weather(**args)
        print(f"Tool called: get_weather({args}) -> {weather}")
        function_outputs.append(
            {
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": json.dumps({"weather": weather}),
            }
        )

print("Step 2: Continue same chain with previous_response_id...")
final = client.responses.create(
    model="openai.gpt-5.4",
    instructions="Answer concisely using the weather information.",
    tools=FUNCTION_TOOLS,
    input=function_outputs,
    previous_response_id=first.id,
)

print("Final response:")
final.output_text


## 6. Built-in Tool: Web Search (`web_search.py`)

Use current web information in the response.


In [ ]:
print("Running web_search-enabled request...")
web_result = client.responses.create(
    model=MODEL_ID,
    tools=[{"type": "web_search"}],
    input="Share one positive business news item from this week.",
)
print("Grounded response:")
web_result.output_text


## 7. Reasoning Settings (`reasoning.py`)

Inspect structured output while controlling reasoning settings.


In [ ]:
print("Running reasoning example...")
reasoning_result = client.responses.create(
    model="openai.gpt-5.4",
    input="What is the answer to 12 * (3 + 9)?",
    reasoning={"effort": "high", "summary": "auto"},
    store=False,
)

print("Showing first 1500 chars of structured output blocks:")
json.dumps(reasoning_result.to_dict()["output"], indent=2)[:1500]


## 8. Optional: Multimodal (`multimodal.py`)

Run this only if sample files exist.

Expected sample path for quick test:
- `output/test_image.png`


In [ ]:
import base64

image_path = Path("output/test_image.png")
if image_path.exists():
    print(f"Using image: {image_path}")
    image_b64 = base64.b64encode(image_path.read_bytes()).decode("utf-8")
    image_response = client.responses.create(
        model=MODEL_ID,
        store=False,
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": "What's in this image?"},
                    {
                        "type": "input_image",
                        "image_url": f"data:image/png;base64,{image_b64}",
                        "detail": "high",
                    },
                ],
            }
        ],
    )
    print("Image response:")
    print(image_response.output_text)
else:
    print("Skip: output/test_image.png not found")


## 9. Optional: Advanced Tools

These mirror `mcp_client.py` and `code_interpreter.py`.
Enable only if your environment has tool access for those capabilities.


In [ ]:
# MCP quick sketch (not executed by default):
# client.responses.create(
#     model="openai.gpt-5.4",
#     tools=[{
#         "type": "mcp",
#         "server_label": "dmcp",
#         "server_url": "https://mcp.deepwiki.com/mcp",
#         "require_approval": "never",
#     }],
#     input="Roll 2d4+1",
# )

# Code interpreter quick sketch (not executed by default):
# client.responses.create(
#     model="openai.gpt-4.1",
#     tools=[{"type": "code_interpreter", "container": {"type": "auto"}}],
#     input="Use python to calculate sqrt(sqrt(4 * 3.82)).",
# )


## 10. Experiment Zone

Try one change at a time and compare results:

- Prompt specificity
- Model ID
- Tool enabled vs disabled
- Reasoning effort level


In [ ]:
experiment_response = client.responses.create(
    model=MODEL_ID,
    input="Explain vector stores in simple terms in 3 bullet points.",
)
experiment_response.output_text


## 11. Practice Exercises

1. Recreate the full `api_state.py` flow with a second tool function.
2. Compare the same prompt across `openai.gpt-4.1` and `openai.gpt-5.2`.
3. Add optional fields to `CalendarEvent` and test parse reliability.
4. Run one answer with and without `web_search` and compare grounding.

## Discussion prompts
- Which pattern felt most predictable in your tests?
- When is manual state chaining better than one-shot prompts?
- Which tool integrations need extra approval controls in your project?
